In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import re
import nltk

In [2]:
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except IndexError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [3]:
data_home = "../input"
output_dir = "../working"

source_file_dir = '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels'
source_file_dir

'/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels'

In [4]:
# This structure matches all the texts
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

# This pattern is common to all the texts
clip_pats_gutenberg = [
    r"\*\*\*\s*START OF",
    r"\*\*\*\s*END OF"
]


# We put some regex character classes into variables 
# for simplicity
roman = '[IVXLCM]+' # All Roman Numbers
caps = "[A-Z';, -]+" # Capital letters and sentence boundaries

# We put the patterns into a list of lists

chap_pats = [
    (345,   rf"^CHAPTER\s+{roman}$"),
    (43,    rf"^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\. JEKYLL WAS QUITE AT EASE|"
            rf"THE CAREW MURDER CASE|INCIDENT OF THE LETTER|INCIDENT OF DR\. LANYON|"
            rf"INCIDENT AT THE WINDOW|THE LAST NIGHT|DR\. LANYON'S NARRATIVE|"
            rf"HENRY JEKYLL'S FULL STATEMENT OF THE CASE)$"),
    (1952,  rf"^The Yellow Wallpaper$"),
]


In [5]:
source_file_list = sorted(glob(f"{source_file_dir}/*.*"))
source_file_list

['/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/GILMAN_CHARLOTTE_THE_YELLOW_WALLPAPER-pg1952.txt',
 '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/STEVENSON_ROBERT_DR_JEKYLL_AND_MR_HYDE-pg43.txt',
 '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/STOKER_BRAM_DRACULA-pg345.txt']

In [6]:
book_data = []
for source_file_path in source_file_list:
    book_id = int(source_file_path.split('-')[-1].split('.')[0].replace('pg',''))
    raw_title = source_file_path.split('/')[-1].split('-')[0].replace('_', ' ')
    book_data.append((book_id, source_file_path, raw_title))

book_data

[(1952,
  '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/GILMAN_CHARLOTTE_THE_YELLOW_WALLPAPER-pg1952.txt',
  'GILMAN CHARLOTTE THE YELLOW WALLPAPER'),
 (43,
  '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/STEVENSON_ROBERT_DR_JEKYLL_AND_MR_HYDE-pg43.txt',
  'STEVENSON ROBERT DR JEKYLL AND MR HYDE'),
 (345,
  '/kaggle/input/datasets/isabellouisedelgado/victorian-gothic-novels/STOKER_BRAM_DRACULA-pg345.txt',
  'STOKER BRAM DRACULA')]

In [7]:
LIB = pd.DataFrame(book_data, columns=['book_id','source_file_path', 'raw_title'])\
    .set_index('book_id').sort_index()
LIB

,source_file_path,raw_title
book_id,,
43,/kaggle/input/datasets/isabellouisedelgado/vic...,STEVENSON ROBERT DR JEKYLL AND MR HYDE
345,/kaggle/input/datasets/isabellouisedelgado/vic...,STOKER BRAM DRACULA
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,GILMAN CHARLOTTE THE YELLOW WALLPAPER


In [8]:
try:
    LIB['author'] = LIB.raw_title.apply(lambda x: ', '.join(x.split()[:2]))
    LIB['title'] = LIB.raw_title.apply(lambda x: ' '.join(x.split()[2:]))
    LIB = LIB.drop('raw_title', axis=1)

# Skip if this cell has already been run
except AttributeError:
    pass

LIB

,source_file_path,author,title
book_id,,,
43,/kaggle/input/datasets/isabellouisedelgado/vic...,"STEVENSON, ROBERT",DR JEKYLL AND MR HYDE
345,/kaggle/input/datasets/isabellouisedelgado/vic...,"STOKER, BRAM",DRACULA
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,"GILMAN, CHARLOTTE",THE YELLOW WALLPAPER


In [9]:
LIB['chap_pat'] = pd.Series({pat[0]:pat[1] for pat in chap_pats})
LIB

,source_file_path,author,title,chap_pat
book_id,,,,
43,/kaggle/input/datasets/isabellouisedelgado/vic...,"STEVENSON, ROBERT",DR JEKYLL AND MR HYDE,^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\...
345,/kaggle/input/datasets/isabellouisedelgado/vic...,"STOKER, BRAM",DRACULA,^CHAPTER\s+[IVXLCM]+$
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,"GILMAN, CHARLOTTE",THE YELLOW WALLPAPER,^The Yellow Wallpaper$


In [10]:
# Add clip_pats and chap_pat to LIB
LIB['clip_pats'] = None
LIB['chap_pat'] = None

for i in LIB.index:
    LIB.at[i, 'clip_pats'] = clip_pats_gutenberg

for book_id, pat in chap_pats:
    LIB.at[book_id, 'chap_pat'] = pat

# Verify it looks right before running corpus
LIB[['title', 'clip_pats', 'chap_pat']]

,title,clip_pats,chap_pat
book_id,,,
43,DR JEKYLL AND MR HYDE,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\...
345,DRACULA,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^CHAPTER\s+[IVXLCM]+$
1952,THE YELLOW WALLPAPER,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^The Yellow Wallpaper$


In [11]:
def tokenize_source(book_id):
    src_file = LIB.loc[book_id].source_file_path
    
    # Convert the raw text into a dataframe
    text_lines = open(src_file, 'r', encoding="utf-8-sig").readlines()
    LINES = pd.DataFrame({'book_id': book_id, 'line_str': text_lines})
    LINES.line_str = LINES.line_str.str.strip()
    
    # Clip the cruft

    clip_pats = LIB.loc[book_id, 'clip_pats']
    start_pat = clip_pats[0]
    end_pat = clip_pats[1]
    start = LINES.line_str.str.contains(start_pat, regex=True)
    end = LINES.line_str.str.contains(end_pat, regex=True)
    try:
        start_line_num = LINES.loc[start].index[0]
    except IndexError:
        raise ValueError("Clip start pattern not found.")
    try:
        end_line_num = LINES.loc[end].index[0]
    except IndexError:
        raise ValueError("Clip end pattern not found.")
    LINES = LINES.loc[start_line_num + 1 : end_line_num - 2].copy()
    
    # Chunk chapters
    chap_pat = LIB.loc[book_id, 'chap_pat']
    chap_lines = LINES.line_str.str.contains(chap_pat, regex=True, case=True)
    LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
    LINES.chap_num = LINES.chap_num.ffill()
    LINES = LINES.loc[~LINES.chap_num.isna()]
    LINES = LINES.loc[~chap_lines]
    LINES.chap_num = LINES.chap_num.astype('int')
    CHAPS = LINES.groupby('chap_num', group_keys=True)['line_str']\
        .apply(lambda x: '\n'.join(x)).to_frame('chap_str')
    del LINES
    
    # Build OHCO records with plain Python loops
    records = []
    for chap_num, chap_str in CHAPS.chap_str.items():
        for para_num, para_str in enumerate(chap_str.split('\n\n')):
            if not para_str.strip():
                continue
            for sent_num, sent_str in enumerate(nltk.sent_tokenize(para_str)):
                for token_num, (token_str, pos) in enumerate(nltk.pos_tag(nltk.word_tokenize(sent_str))):
                    records.append((chap_num, para_num, sent_num, token_num, token_str, pos))
    del CHAPS
    
    # Build TOKENS dataframe
    TOKENS = pd.DataFrame(records, columns=['chap_num', 'para_num', 'sent_num', 'token_num', 'token_str', 'pos'])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex=True)
    TOKENS = TOKENS[TOKENS.term_str != ''].copy()
    TOKENS = TOKENS.set_index(['chap_num', 'para_num', 'sent_num', 'token_num'])
    
    return TOKENS

In [12]:
corpus = []
for i in LIB.index:
    print(LIB.loc[i].title)
    corpus.append(tokenize_source(i))
print("Done")
CORPUS = pd.concat(corpus, keys=LIB.index)
CORPUS.index.names = OHCO
del corpus

DR JEKYLL AND MR HYDE
DRACULA
THE YELLOW WALLPAPER
Done


In [13]:
CORPUS

token_str  pos pos_group  \
book_id chap_num para_num sent_num token_num                             
43      8        0        0        0                DR.  NNP        NN   
                                   1             LANYON  NNP        NN   
                                   3                  S  NNP        NN   
                                   4          NARRATIVE  NNP        NN   
                 1        0        0              HENRY  NNP        NN   
...                                                 ...  ...       ...   
1952    1        267      1        18             creep   VB        VB   
                                   19              over   IN        IN   
                                   20               him  PRP        PR   
                                   21             every   DT        DT   
                                   22              time   NN        NN   

                                               term_str  
book_id chap_num para_num sent_num token_num             
43      8        0        0        0                 dr  
                                   1             lanyon  
                                   3                  s  
                                   4          narrative  
                 1        0        0              henry  
...                                                 ...  
1952    1        267      1        18             creep  
                                   19              over  
                                   20               him  
                                   21             every  
                                   22              time  

[194368 rows x 4 columns]

In [14]:
LIB.to_csv('/kaggle/working/LIB_victorian_gothic.csv')
CORPUS.to_csv('/kaggle/working/CORPUS_victorian_gothic.csv')